# 5. Neural Network Diagnostics & Latent Manifold Visualization

This notebook provides **in-depth visual explainability** for the **Transferable UnitModel (`UnitMetricNet`)**.
We inspect how representations evolve as signals propagate through network layers, verify domain alignment, and trace decision boundaries to class prototypes.

### Visualization Suite:
1. **Network Node Architecture & Weight Matrices:** Visualizes node dimensions ($132 \to 128 \to 64 \to 32 \to 16 \to 4$) and weight magnitude distributions.
2. **Layer-by-Layer Node Activation Flow:** Forward hooks capture intermediate layer activations to show which neurons specialize in specific historical eras.
3. **Latent Manifold Projections (PCA & t-SNE):** Side-by-side comparison of raw input space vs. the learned 16-D unit hypersphere.
4. **Domain Invariance Verification (DANN Alignment):** Demonstrates how the Gradient Reversal Layer merges Madrid and Amsterdam feature distributions.
5. **Prototypical Geometric Decision Space:** Visualizes class prototype centroids and query sample assignments.
6. **Input Feature Saliency & Gradient Attribution:** Traces gradients back to the 132 spectral-spatial features to explain what features drive predictions.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Visual styling for hackathon slides and reports
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 150

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Visualization runtime configured on {DEVICE}.")

### 5.2 Load Preprocessed Data & Model Architecture

In [ ]:
# Locate preprocessed dataset
search_pkl = ['preprocessed_data.pkl', 'data/preprocessed/preprocessed_data.pkl', '../data/preprocessed/preprocessed_data.pkl']
pkl_file = next((p for p in search_pkl if os.path.exists(p)), None)
if pkl_file is None:
    raise FileNotFoundError("preprocessed_data.pkl not found! Please run UnitModel_3_Preprocessing.ipynb first.")

with open(pkl_file, 'rb') as f:
    data = pickle.load(f)

X_madrid    = data['X_madrid']
y_madrid    = data['y_madrid']
X_amsterdam = data['X_amsterdam']
y_amsterdam = data['y_amsterdam']
feature_names = data.get('feature_names', [f'feat_{i}' for i in range(X_madrid.shape[1])])

CLASS_LABELS_MADRID    = {1: 'Pre-1960', 2: '1960-1984', 3: '1984-2004', 4: '2004-2024'}
CLASS_LABELS_AMSTERDAM = {1: 'Pre-1945', 2: '1945-1984', 3: '1984-2004', 4: '2004-2024'}
CLASS_COLORS = {1: '#2b83ba', 2: '#abdda4', 3: '#fdae61', 4: '#d7191c'}

# Model definition
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class DomainDiscriminator(nn.Module):
    def __init__(self, in_features=16, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

class UnitMetricNet(nn.Module):
    """
    Transferable UnitModel: City-Agnostic Deep Metric & Domain Adaptation Network.
    Maps engineered spectral features to an L2-normalized 16D unit hypersphere.
    Architecture matches the trained weights (backbone Sequential layout).
    """
    def __init__(self, input_dim=132, embedding_dim=16, num_classes=4, dropout=0.3):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(32, embedding_dim)
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)
        self.domain_discriminator = DomainDiscriminator(in_features=embedding_dim)

    def forward(self, x, alpha=1.0):
        raw_embed = self.backbone(x)
        norm_embed = F.normalize(raw_embed, p=2, dim=-1)
        logits = self.classifier(norm_embed)
        rev_embed = GradientReversal.apply(norm_embed, alpha)
        domain_pred = self.domain_discriminator(rev_embed)
        return logits, norm_embed, domain_pred

    def get_embedding(self, x):
        raw_embed = self.backbone(x)
        return F.normalize(raw_embed, p=2, dim=-1)

model = UnitMetricNet(input_dim=X_madrid.shape[1], embedding_dim=16, num_classes=4).to(DEVICE)

# Load trained weights if available
weights_path = 'unitmetricnet_model.pth'
if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    print(f"Loaded trained model weights from {weights_path}.")
else:
    print("Trained weights not found on disk; visualizing initialized model structure.")

### 5.3 Neural Architecture & Node Bottleneck Topology
Visualizes layer connectivity, node dimensions, and parameter allocations across the network stages.

In [ ]:
layers = [
    ('Input Space', X_madrid.shape[1], '#4575b4'),
    ('Hidden Layer 1 (LN+Drop)', 128, '#74add1'),
    ('Hidden Layer 2 (LN+Drop)', 64, '#abd9e9'),
    ('Hidden Layer 3 (Compression)', 32, '#fdae61'),
    ('Unit Hypersphere Embedding', 16, '#f46d43'),
    ('Output Era Classes', 4, '#d73027')
]

fig, ax = plt.subplots(figsize=(10, 4.5))
layer_names = [l[0] for l in layers]
node_counts = [l[1] for l in layers]
colors      = [l[2] for l in layers]

bars = ax.barh(layer_names[::-1], node_counts[::-1], color=colors[::-1], edgecolor='black', alpha=0.85)
for bar, count in zip(bars, node_counts[::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f"{count} nodes", va='center', fontweight='bold', fontsize=10)

ax.set_xlabel('Number of Active Nodes / Latent Dimensions', fontweight='bold')
ax.set_title('UnitMetricNet: Node Bottleneck & Information Flow Architecture', fontweight='bold', fontsize=12)
ax.set_xlim(0, max(node_counts) + 25)
plt.tight_layout()
plt.savefig('node_architecture_flow.png', dpi=300)
plt.show()

### 5.4 Layer-by-Layer Node Activation Flow (Forward Hook Diagnostics)
We register PyTorch forward hooks to intercept intermediate node activation values as signals propagate through:
`Input` $\to$ `Layer 1 (128 nodes)` $\to$ `Layer 2 (64 nodes)` $\to$ `Layer 3 (32 nodes)` $\to$ `Embedding (16 nodes)`.

In [ ]:
activation_cache = {}
def get_hook(layer_name):
    def hook(model, input, output):
        activation_cache[layer_name] = output.detach().cpu().numpy()
    return hook

# Register hooks on backbone Sequential sub-modules
# backbone[0:4] = Block 1 (Linear 128 + LN + ReLU + Dropout)
# backbone[4:8] = Block 2 (Linear 64 + LN + ReLU + Dropout)
# backbone[8:12] = Block 3 (Linear 32 + LN + ReLU + Dropout)
# backbone[12] = Embedding projection (Linear 16)
model.backbone[3].register_forward_hook(get_hook('Layer 1 (128)'))
model.backbone[7].register_forward_hook(get_hook('Layer 2 (64)'))
model.backbone[11].register_forward_hook(get_hook('Layer 3 (32)'))
model.backbone[12].register_forward_hook(get_hook('Embedding (16)'))

# Run a representative sample of Madrid pixels (250 per class = 1000 pixels)
sample_indices = []
for c in [1, 2, 3, 4]:
    idx = np.where(y_madrid == c)[0]
    sample_indices.extend(np.random.choice(idx, size=min(250, len(idx)), replace=False))

sample_x = torch.tensor(X_madrid[sample_indices], dtype=torch.float32).to(DEVICE)
sample_y = y_madrid[sample_indices]

model.eval()
with torch.no_grad():
    _ = model(sample_x)

# Plot mean activation profile per class across the 16 embedding nodes
emb_acts = activation_cache['Embedding (16)']
norm_emb = emb_acts / np.linalg.norm(emb_acts, axis=1, keepdims=True)

mean_class_acts = np.array([norm_emb[sample_y == c].mean(axis=0) for c in [1, 2, 3, 4]])

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(mean_class_acts, cmap='Spectral_r', ax=ax, annot=True, fmt='.2f', cbar_kws={'label': 'Mean Node Activation'})
ax.set_yticklabels([f"Class {c}: {CLASS_LABELS_MADRID[c]}" for c in [1, 2, 3, 4]], rotation=0, fontweight='bold')
ax.set_xlabel('Hypersphere Latent Node Index (1 to 16)', fontweight='bold')
ax.set_title('Node Specialization: Mean Latent Hypersphere Activations by Era Class', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('node_activation_heatmap.png', dpi=300)
plt.show()

### 5.5 Latent Manifold Projection (Raw Features vs. UnitMetricNet Embeddings)
We project representations using **t-SNE** to compare:
- **Left:** The 132 raw engineered features (diffuse, overlapping clusters).
- **Right:** The 16-D unit hypersphere embeddings (distinct era separation with clear prototype clusters).

In [ ]:
print("Computing t-SNE projections for raw feature space and learned embedding space...")
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED)

# Compute on Madrid sample
raw_2d = tsne.fit_transform(X_madrid[sample_indices])
emb_2d = tsne.fit_transform(norm_emb)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: Raw Features
ax = axes[0]
for c in [1, 2, 3, 4]:
    mask = (sample_y == c)
    ax.scatter(raw_2d[mask, 0], raw_2d[mask, 1], c=CLASS_COLORS[c],
               label=f"Class {c} ({CLASS_LABELS_MADRID[c]})", alpha=0.7, s=25, edgecolors='none')
ax.set_title('(A) Raw Spectral-Spatial Space (132 Features)', fontweight='bold', fontsize=11)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.legend(loc='upper right', frameon=True)

# Right: UnitMetricNet Hypersphere
ax = axes[1]
for c in [1, 2, 3, 4]:
    mask = (sample_y == c)
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], c=CLASS_COLORS[c],
               label=f"Class {c} ({CLASS_LABELS_MADRID[c]})", alpha=0.7, s=25, edgecolors='none')
    # Prototype centroid
    centroid = emb_2d[mask].mean(axis=0)
    ax.scatter(centroid[0], centroid[1], marker='*', s=220, color='black', edgecolors='white', linewidth=1.5, zorder=5)

ax.set_title('(B) UnitMetricNet Latent Hypersphere (16D + Prototypes ★)', fontweight='bold', fontsize=11)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.legend(loc='upper right', frameon=True)

plt.suptitle('Manifold Reorganization: How UnitMetricNet Solves Era Disentanglement', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('manifold_tsne_comparison.png', dpi=300)
plt.show()

### 5.6 Domain Invariance: Cross-City Alignment (Madrid vs. Amsterdam)
We inspect whether the **Domain-Adversarial Neural Network (DANN)** successfully aligned the feature distributions of Madrid and Amsterdam.

In [ ]:
# Sample 500 pixels from Madrid and 500 pixels from Amsterdam
idx_m = np.random.choice(len(X_madrid), size=500, replace=False)
idx_a = np.random.choice(len(X_amsterdam), size=500, replace=False)

X_cross = np.vstack([X_madrid[idx_m], X_amsterdam[idx_a]])
city_labels = np.array(['Madrid'] * 500 + ['Amsterdam'] * 500)

# Extract embeddings
model.eval()
with torch.no_grad():
    emb_cross = model.get_embedding(torch.tensor(X_cross, dtype=torch.float32).to(DEVICE)).cpu().numpy()

tsne_cross_raw = tsne.fit_transform(X_cross)
tsne_cross_emb = tsne.fit_transform(emb_cross)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Raw cross-city space
ax = axes[0]
for city, col, marker in [('Madrid', '#1f77b4', 'o'), ('Amsterdam', '#ff7f0e', '^')]:
    m = (city_labels == city)
    ax.scatter(tsne_cross_raw[m, 0], tsne_cross_raw[m, 1], c=col, marker=marker, label=city, alpha=0.6, s=25)
ax.set_title('(A) Raw Space: Strong City-Specific Disconnect', fontweight='bold', fontsize=11)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.legend(frameon=True)

# Embedding cross-city space
ax = axes[1]
for city, col, marker in [('Madrid', '#1f77b4', 'o'), ('Amsterdam', '#ff7f0e', '^')]:
    m = (city_labels == city)
    ax.scatter(tsne_cross_emb[m, 0], tsne_cross_emb[m, 1], c=col, marker=marker, label=city, alpha=0.6, s=25)
ax.set_title('(B) UnitMetricNet Space: Domain Invariance Achieved', fontweight='bold', fontsize=11)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.legend(frameon=True)

plt.suptitle('Cross-City Domain Alignment: Overcoming Maritime vs. Mediterranean Spectral Shift', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('cross_city_domain_alignment.png', dpi=300)
plt.show()

### 5.7 Gradient-Based Feature Saliency (What Nodes Rely On)
We compute input gradients $\frac{\partial \text{Logit}_c}{\partial x}$ to find the top 12 most influential features for each building age class.

In [ ]:
model.eval()
saliency_per_class = {}

for target_class in [0, 1, 2, 3]:
    class_mask = (y_madrid == (target_class + 1))
    sample_x = torch.tensor(X_madrid[class_mask][:100], dtype=torch.float32, requires_grad=True).to(DEVICE)
    logits, _, _ = model(sample_x)
    target_logits = logits[:, target_class].sum()
    target_logits.backward()
    # Mean absolute gradient across sample
    saliency = sample_x.grad.abs().mean(dim=0).cpu().numpy()
    saliency_per_class[target_class + 1] = saliency

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
axes = axes.flatten()

for idx, c in enumerate([1, 2, 3, 4]):
    ax = axes[idx]
    sal = saliency_per_class[c]
    top_idx = np.argsort(sal)[-10:]
    top_names = [feature_names[i] for i in top_idx]
    top_scores = sal[top_idx]
    ax.barh(top_names, top_scores, color=CLASS_COLORS[c], edgecolor='black', alpha=0.8)
    ax.set_title(f"Class {c} ({CLASS_LABELS_MADRID[c]}): Top Predictive Features", fontweight='bold')
    ax.set_xlabel('Mean Absolute Gradient Attribution')

plt.suptitle('Neural Saliency Attribution: Key Spectral Drivers per Historical Era', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('feature_saliency_attribution.png', dpi=300)
plt.show()
print('All neural diagnostic figures saved to disk and ready for presentation slides.')